In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import os

In [ ]:
image_urls = {
    "image1.jpg": "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg",
    "image2.jpg": "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/baboon.jpg",
    "image3.jpg": "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/pears.png"
}

images_raw = []
for name, url in image_urls.items():
    try:
        urllib.request.urlretrieve(url, name)
        img = cv2.imread(name)
        if img is None:
            raise ValueError
    except Exception:
        if name == "image1.jpg":
            img = np.zeros((300, 300, 3), dtype=np.uint8)
            cv2.circle(img, (150, 150), 80, (0, 0, 255), -1)
            cv2.rectangle(img, (50, 50), (250, 250), (255, 0, 0), 10)
        elif name == "image2.jpg":
            img = np.zeros((300, 300, 3), dtype=np.uint8)
            for y in range(300):
                img[y, :, 1] = int(y / 300 * 255)
            cv2.putText(img, "Synthetic", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 3)
        else:
            img = np.zeros((300, 300, 3), dtype=np.uint8)
            img[:, :, 2] = 128
            cv2.line(img, (0, 0), (300, 300), (0, 255, 0), 15)
    images_raw.append(img)

def standardise(img):
    img_resized = cv2.resize(img, (224, 224))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    img_norm = img_rgb.astype(np.float32) / 255.0
    return img_norm

processed_images = [standardise(img) for img in images_raw]

In [ ]:
denoised_gaussian_list = []
denoised_nlm_list = []
psnr_gaussian_list = []
psnr_nlm_list = []

for img_norm in processed_images:
    img_uint8 = (img_norm * 255).astype(np.uint8)
    g_blur = cv2.GaussianBlur(img_uint8, (5, 5), 0)
    nlm = cv2.fastNlMeansDenoisingColored(img_uint8, None, 10, 10, 7, 21)

    psnr_g = cv2.PSNR(img_uint8, g_blur)
    psnr_n = cv2.PSNR(img_uint8, nlm)

    denoised_gaussian_list.append(g_blur.astype(np.float32) / 255.0)
    denoised_nlm_list.append(nlm.astype(np.float32) / 255.0)
    psnr_gaussian_list.append(psnr_g)
    psnr_nlm_list.append(psnr_n)

In [ ]:
np.random.seed(42)
augmented_images_list = []

for img_norm in processed_images:
    aug_flip = cv2.flip(img_norm, 1)

    img_uint8 = (img_norm * 255).astype(np.uint8)
    hsv = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2HSV)
    h, s, v = cv2.split(hsv)
    v = np.clip(v.astype(np.float32) * 1.3, 0, 255).astype(np.uint8)
    hsv_bright = cv2.merge([h, s, v])
    rgb_bright = cv2.cvtColor(hsv_bright, cv2.COLOR_HSV2RGB)
    aug_bright = rgb_bright.astype(np.float32) / 255.0

    aug_noise = np.clip(img_norm + np.random.normal(0, 0.05, img_norm.shape), 0.0, 1.0)

    h_dim, w_dim, _ = img_norm.shape
    y = np.random.randint(0, h_dim - 180)
    x = np.random.randint(0, w_dim - 180)
    cropped = img_norm[y:y+180, x:x+180]
    aug_crop = cv2.resize(cropped, (224, 224))

    augmented_images_list.append({
        "flip": aug_flip,
        "bright": aug_bright,
        "noise": aug_noise,
        "crop": aug_crop
    })

In [ ]:
chosen_idx = 0
orig = processed_images[chosen_idx]
g_denoised = denoised_gaussian_list[chosen_idx]
n_denoised = denoised_nlm_list[chosen_idx]
augs = augmented_images_list[chosen_idx]

fig, axes = plt.subplots(3, 4, figsize=(15, 11))

axes[0, 0].imshow(orig)
axes[0, 0].set_title("Original")
axes[0, 1].imshow(g_denoised)
axes[0, 1].set_title("Gaussian Denoised")
axes[0, 2].imshow(n_denoised)
axes[0, 2].set_title("NLM Denoised")
axes[0, 3].axis("off")

axes[1, 0].imshow(augs["flip"])
axes[1, 0].set_title("Aug: Flip")
axes[1, 1].imshow(augs["bright"])
axes[1, 1].set_title("Aug: Brightness")
axes[1, 2].imshow(augs["noise"])
axes[1, 2].set_title("Aug: Noise")
axes[1, 3].imshow(augs["crop"])
axes[1, 3].set_title("Aug: Crop & Resize")

def get_canny(img):
    gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    return cv2.Canny(gray, 100, 200)

axes[2, 0].imshow(get_canny(orig), cmap="gray")
axes[2, 0].set_title("Canny: Original")
axes[2, 1].imshow(get_canny(augs["flip"]), cmap="gray")
axes[2, 1].set_title("Canny: Flip")
axes[2, 2].imshow(get_canny(augs["bright"]), cmap="gray")
axes[2, 2].set_title("Canny: Brightness")
axes[2, 3].imshow(get_canny(augs["crop"]), cmap="gray")
axes[2, 3].set_title("Canny: Crop")

for i in range(3):
    for j in range(4):
        if not (i == 0 and j == 3):
            axes[i, j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("| Image | PSNR (Gaussian) | PSNR (NLM) | Mean Pixel | Std Pixel | Sharpness |")
print("|---|---|---|---|---|---|")
for i, orig in enumerate(processed_images):
    g_psnr = psnr_gaussian_list[i]
    n_psnr = psnr_nlm_list[i]
    mean_val = np.mean(orig)
    std_val = np.std(orig)

    gray = cv2.cvtColor((orig * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()

    print(f"| Image {i+1} | {g_psnr:.2f} dB | {n_psnr:.2f} dB | {mean_val:.4f} | {std_val:.4f} | {sharpness:.2f} |")

### Reflection

#### 1. Why is normalising to [0, 1] important before feeding images to a neural network?
- **Numerical Stability**: Restricting pixel values to a small, bounded range (such as [0, 1]) prevents activation values in deep layers from growing too large, which helps avoid exploding gradient issues.
- **Optimization Speed**: Standardized inputs yield a more symmetrical loss surface, allowing gradient descent algorithms to converge significantly faster and more reliably.

#### 2. Which denoising method preserved quality better based on PSNR?
- **Non-Local Means (NLM)** typically preserves image structure and detail much better than Gaussian Blur. Gaussian Blur uniformly attenuates high frequencies, smoothing out sharp edges and fine textures, which results in a lower PSNR relative to the original. NLM searches for similar patches across the image to perform denoising, preserving edges better and yielding a higher PSNR.

#### 3. Why is data augmentation useful even when you have a large dataset?
- **Improved Generalization**: It simulates real-world variations (e.g., changes in lighting, rotation, framing, sensor noise) that might not be fully represented in the static dataset.
- **Regularization**: Augmentation serves as a powerful regularizer, preventing the neural network from memorizing training patterns and forcing it to learn invariant features (e.g., shape features instead of absolute brightness or exact position).